# Qwen P2: probe balanced accuracy by loudness decile

Held-out 72. One row per (trajectory, step, reasoning token), every probe read on **every**
token, so no selection sits between the loudness axis and the label and the axis spans the
real distribution.

**Input** is `score_probes_heldout_per_token.sh`'s table. It carries *both* lenses' loudness
columns side by side (`score_probes_per_token.py` loops over `jlens` and `logitlens` with no
flag), which is what lets cell 5 re-bin the same rows on logitlens loudness without a second
GPU pass. Everything except cell 5 bins on J-lens.

**The statistics are the repo's, not this notebook's.** `stats.bal_acc` (mean per-class recall
over the classes present in the bin), `stats.boot_bal_acc` (95% CI resampling **trajectory
names**, never rows -- tokens in a chain share a label and a sentence structure, so a
row-level bootstrap reports bands several times too narrow) and `stats.qbin` (quantile bins
that survive ties). `columns.axis_label` owns the loudness axis label, so these panels sit
beside the pipeline's own figures. Do **not** reach for `plotting/_style.bal_acc` here: its
`ACTIONS` is a module constant and it returns a silent `nan` off the four-action label.

**Which label.** `score_probes_per_token.py` labels a token with its step's `agent_action` --
where the trajectory *ended up*. These probes were trained on the **local belief** (what the
model answers if reasoning stops at that token), so this measures them against the final
action. Point `TABLE` at `join_rollouts.py`'s output instead and cell 1 picks up `label_local`
automatically; the rest of the notebook is unchanged.

## 1. Imports and load

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from telos_interp.loudness_analysis import columns as cols
from telos_interp.loudness_analysis import stats

# ---- what this run is ---------------------------------------------------------------
TABLE = Path("/workspace/results/qwen_p2_local_belief/heldout/per_token_scores.csv")
FIG_DIR = TABLE.parent / "figures"
LAYER = 27
SIGNAL = "direction"
N_DECILES = 10
N_BOOT = 300
SEED = 42

FIG_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid", context="notebook")

# keep_default_na=False because a decoded token can literally be the string "NA", which
# pandas' NA handling turns into a missing value and silently corrupts the token column.
# The numeric columns are coerced explicitly below instead.
df = pd.read_csv(TABLE, keep_default_na=False, na_values=[""], low_memory=False)
print(f"{len(df):,} token rows, {df['name'].nunique()} trajectories")

# ---- which column is the truth ------------------------------------------------------
# label_local exists only in join_rollouts.py's output (action NAMES); the per-token
# scorer writes the step's final action as `label` (integer ids, LEFT/UP/RIGHT/DOWN = 0-3).
TRUTH = "label_local" if "label_local" in df.columns else "label"
df = df[df[TRUTH].astype(str) != ""]
CLASSES = sorted(df[TRUTH].unique())
print(f"truth column: {TRUTH}   classes: {CLASSES}")

# ---- which probes are in the table --------------------------------------------------
# Discovered from the columns, never hard-coded: score_probes_per_token.py keys a probe
# "<parent dir>.<file stem>", so the keys carry the probe directory's name too.
PROBE_KEYS = [c[: -len("_pred")] for c in df.columns if c.endswith("_pred")]


def short(key: str) -> str:
    """`...qwen_p2_local_belief_jlens_l27_lr` -> `jlens lr`."""
    m = re.search(r"(?P<arm>[A-Za-z]+)_l\d+_(?P<kind>lr|mlp)$", key)
    return f"{m['arm']} {m['kind']}" if m else key.rsplit(".", 1)[-1]


LABEL = {k: short(k) for k in PROBE_KEYS}
MLP = [k for k in PROBE_KEYS if LABEL[k].endswith("mlp")]

# One colour per selection arm, one linestyle per probe family, so an arm keeps its colour
# across every panel below.
ARMS = sorted({LABEL[k].split()[0] for k in PROBE_KEYS})
COLOR = dict(zip(ARMS, sns.color_palette("colorblind", len(ARMS)), strict=True))
STYLE = {"lr": "-", "mlp": "--"}

for k in PROBE_KEYS:
    print(f"  {LABEL[k]:<16} {k}")
print(f"\nMLP probes: {[LABEL[k] for k in MLP]}")

# The loudness columns present, one per lens.
for lens in ("jlens", "logitlens"):
    print(f"  {lens}: {cols.resolve(df.columns, lens, SIGNAL, LAYER)}")

## 2. Aggregate by loudness decile

In [ ]:
def with_decile(frame: pd.DataFrame, lens: str) -> pd.DataFrame:
    """Add `loudness` and a 1..10 `decile` for `lens`, dropping tokens with no mass cell.

    Deciles are re-cut per lens on purpose: the two lenses' top-20 sets overlap only about
    half, so decile 10 under one is not decile 10 under the other.
    """
    col = cols.resolve(frame.columns, lens, SIGNAL, LAYER)
    d = frame.copy()
    d["loudness"] = pd.to_numeric(d[col], errors="coerce")
    d = d[np.isfinite(d["loudness"])]
    d["decile"] = stats.qbin(d["loudness"], N_DECILES, labels=False).astype(int) + 1
    return d


def decile_table(frame: pd.DataFrame, lens: str, n_boot: int = N_BOOT) -> pd.DataFrame:
    """Balanced accuracy per (probe, decile), with a trajectory-clustered 95% CI."""
    d = with_decile(frame, lens)
    rng = np.random.default_rng(SEED)
    rows = []
    for dec, g in d.groupby("decile", observed=True):
        for key in PROBE_KEYS:
            point, lo, hi = stats.boot_bal_acc(g, TRUTH, f"{key}_pred", n_boot, rng, CLASSES)
            rows.append(
                {
                    "lens": lens,
                    "decile": int(dec),
                    "probe": LABEL[key],
                    "probe_key": key,
                    "bal_acc": point,
                    "lo": lo,
                    "hi": hi,
                    "n": len(g),
                    "n_traj": g["name"].nunique(),
                    "mean_logmass": float(g["loudness"].mean()),
                }
            )
    return pd.DataFrame(rows)


BY_DECILE = pd.concat([decile_table(df, "jlens"), decile_table(df, "logitlens")], ignore_index=True)
BY_DECILE.to_csv(TABLE.parent / "accuracy_by_loudness_decile.csv", index=False)

# The bin sizes are worth a look before the panels: with 72 trajectories a decile can be
# dominated by a handful of chains, and that is what the CI is reporting.
print(
    BY_DECILE[BY_DECILE.lens == "jlens"]
    .drop_duplicates("decile")[["decile", "n", "n_traj", "mean_logmass"]]
    .to_string(index=False)
)
BY_DECILE.head()

## 3. One panel per probe (J-lens loudness)

In [ ]:
CHANCE = 1.0 / len(CLASSES)


def draw(ax, tab: pd.DataFrame, keys, lens: str) -> None:
    """Loudness decile on x, balanced accuracy on y, with the clustered 95% band."""
    for key in keys:
        t = tab[(tab.probe_key == key) & (tab.lens == lens)].sort_values("decile")
        arm, kind = LABEL[key].split()
        err = np.vstack(
            [
                np.clip(t.bal_acc - t.lo, 0, None).fillna(0),
                np.clip(t.hi - t.bal_acc, 0, None).fillna(0),
            ]
        )
        ax.errorbar(
            t.decile,
            t.bal_acc,
            yerr=err,
            marker="o",
            ms=4,
            lw=1.6,
            ls=STYLE.get(kind, "-"),
            color=COLOR[arm],
            capsize=2,
            label=LABEL[key],
        )
    ax.axhline(CHANCE, ls=":", lw=1, color="0.4", label=f"chance ({CHANCE:.2f})")
    ax.set_xlabel(f"{cols.axis_label(lens, SIGNAL, LAYER)} decile\n(1 = quietest, {N_DECILES} = loudest)")
    ax.set_ylabel("Balanced accuracy")
    ax.set_xticks(sorted(tab.decile.unique()))


def save(fig, name: str) -> None:
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"{name}.png", dpi=160)
    print(f"-> {FIG_DIR / f'{name}.png'}")


for key in PROBE_KEYS:
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    draw(ax, BY_DECILE, [key], "jlens")
    ax.set_title(f"{LABEL[key]} -- held-out 72, every reasoning token")
    ax.legend(loc="upper left", fontsize=8)
    save(fig, f"decile_jlens_{LABEL[key].replace(' ', '_')}")
    plt.show()

## 4. All MLP probes, J-lens loudness

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.0))
draw(ax, BY_DECILE, MLP, "jlens")
ax.set_title("MLP probes by J-lens direction loudness decile")
ax.legend(loc="upper left", fontsize=9)
save(fig, "decile_jlens_all_mlp")
plt.show()

## 5. All MLP probes, logitlens loudness

The same rows and the same probes, re-cut into deciles of **logitlens** loudness. Any
difference from cell 4 is the ruler and nothing else -- which is the point: at a single
layer the two lenses' top-20 sets overlap only about half, so "loud" is not one quantity.

In [ ]:
fig, ax = plt.subplots(figsize=(6.4, 5.0))
draw(ax, BY_DECILE, MLP, "logitlens")
ax.set_title("MLP probes by logitlens direction loudness decile")
ax.legend(loc="upper left", fontsize=9)
save(fig, "decile_logitlens_all_mlp")
plt.show()